In [1]:

import jax 
import jax.numpy as jnp
from jax.tree_util import register_pytree_node_class, PyTreeDef

from jaxtyping import Array
from jax.random import KeyArray

from functools import partial
from itertools import accumulate

@register_pytree_node_class
class MCMCState:
    """MCMC state object."""

    def __init__(
        self, key: KeyArray, x: Array, stats: dict = {}, params: dict = {}
    ) -> None:
        """This class represents the state of the MCMC chain.

        Args:
            key (KeyArray): A random number generator key.
            x (Array): Current value of the chain.
            stats (dict, optional): Statistics that are tracked for the chain. Defaults to {}.
            params (dict, optional): Adaptive parameters for the MCMCKernel. Defaults to {}.
        """
        self.key = key
        self.x = x
        self.params = params
        self.stats = stats

    def __repr__(self) -> str:
        return f"MCMCState(key={self.key}, x={self.x})"

    # Jax stuff
    def tree_flatten(self):
        return (self.key, self.x, self.stats, self.params), None

    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children)

In [11]:
from jax.scipy import stats 

stats.expon.rvs(size=10)

AttributeError: module 'jax.scipy.stats.expon' has no attribute 'rvs'

In [2]:
def flatten_potential_fn(potential_fn, in_vals):
    """Process the potential function to return a function that takes in a single argument."""
    leaves, in_tree = jax.tree_util.tree_flatten(in_vals)
    shapes = tuple(jax.tree_map(lambda x: jnp.shape(x), leaves))
    lengths = jax.tree_map(lambda x: jnp.size(x), leaves)
    cum_lengths = tuple(accumulate(lengths))[:-1]

    def _flatten(x):
        leaves, _ = jax.tree_util.tree_flatten(x)
        flatten_leaves = jax.tree_map(lambda x: jnp.ravel(x), leaves)
        return jnp.concatenate(flatten_leaves)
    
    @partial(jax.jit, static_argnums=(1,2))
    def _unflatten(x, cum_lengths, shapes):
        flattened_leaves = jnp.split(x, cum_lengths)
        leaves = jax.tree_map(lambda x, s: jnp.reshape(x, s), tuple(flattened_leaves), shapes)
        return jax.tree_util.tree_unflatten(in_tree, leaves)

    def _flatten_potential_fn(x):
        x = _unflatten(x, cum_lengths, shapes)
        return potential_fn(*x)
    return _flatten, _unflatten, _flatten_potential_fn


def sliced_potential_fn(flatten_potential_fn, loc, direction):
    """Returns a function that slices the potential function in a given direction."""
    def _sliced_potential_fn(t):
        return flatten_potential_fn(loc + t * direction)
    return _sliced_potential_fn

def conditional_potential_fn(flatten_potential_fn, x, indices):
    """Returns a function that slices the potential function in a given direction."""
    def _conditional_potential_fn(sub_x):
        return flatten_potential_fn(x.at[indices].set(sub_x))
    return _conditional_potential_fn


In [46]:
jnp.upda

ImportError: cannot import name 'index_update' from 'jax.ops' (/root/miniconda3/envs/rvtorch/lib/python3.10/site-packages/jax/ops/__init__.py)

In [43]:
@jax.jit
def f():
    x = jnp.ones(3)
    for i in range(10):
        f = sliced_potential_fn(_potential_fn,x, x)
        x = x + f(i)

    return x

Array([5.2827184e+11, 5.2827184e+11, 5.2827184e+11], dtype=float32)

In [30]:
def log_potential(x, y, z):
    return x+y+z



_flatten, _unflatten, _potential_fn = flatten_potential_fn(log_potential, (0.,1.,2.))
x, y, z = jnp.array([0.]), jnp.array([1.]), jnp.array([2.])
xyz = jnp.concatenate([x,y,z])

print(log_potential(x,y,z), _potential_fn(xyz))


[3.] 3.0


In [99]:
%%timeit
jax.jit(log_potential)(x,y,z)

69.9 µs ± 5.45 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [100]:
%%timeit
jax.jit(_potential_fn)(xyz)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'

In [30]:
vals = {"X": jnp.ones((1,2)), "Y": jnp.ones((1,2))}
leaves, in_tree = jax.tree_util.tree_flatten(vals)
flatten_leaves = jax.tree_map(lambda x: jnp.ravel(x), leaves)
concat_leaves = jnp.concatenate(flatten_leaves)
shapes = jax.tree_map(lambda x: jnp.shape(x), leaves)
lengths = jax.tree_map(lambda x: jnp.size(x), leaves)
cum_lengths = jnp.cumsum(jnp.array(lengths))[:-1]
#jnp.split(jnp.ones(3), jnp.cumsum(jnp.array([x.size for x in shapes])[:-1]))

In [32]:
jnp.split(concat_leaves, cum_lengths)

[Array([1., 1.], dtype=float32),
 Array([1., 1.], dtype=float32),
 Array([], dtype=float32)]

In [396]:
jax.tree_map(lambda x: print(x), states, is_leaf=lambda x: not isinstance(x, MCMCState))

{'X': (Array([[ 7.644402  ],
       [ 5.614325  ],
       [ 0.77242935],
       [ 3.9256144 ],
       [ 0.50295746],
       [ 0.8830856 ],
       [-2.6073875 ],
       [-1.7663181 ],
       [ 0.7013783 ],
       [-0.10531031],
       [ 1.6350682 ],
       [ 2.4129593 ],
       [ 0.35808203],
       [ 3.1471562 ],
       [ 1.9962252 ],
       [ 4.3996115 ],
       [-0.7749269 ],
       [ 1.610878  ],
       [ 0.7561169 ],
       [ 1.5065299 ],
       [ 4.3829355 ],
       [-0.87187356],
       [-4.656183  ],
       [-2.50939   ],
       [-2.5039155 ],
       [ 4.0827394 ],
       [-1.2554551 ],
       [ 1.8207831 ],
       [-3.3005621 ],
       [ 0.7590974 ],
       [-4.906717  ],
       [-6.6593723 ],
       [-1.1078409 ],
       [-1.6584114 ],
       [-1.2116597 ],
       [ 7.015596  ],
       [ 4.93051   ],
       [-4.2398562 ],
       [ 7.860724  ],
       [-2.1599095 ],
       [ 2.03363   ],
       [ 1.5277879 ],
       [-2.4128425 ],
       [ 4.4013124 ],
       [-5.4367805 ],
   

In [403]:
import jax 
import jax.numpy as jnp
import jax.random as jrandom

from functools import partial


key = jrandom.PRNGKey(0)
vars = {"X": jnp.ones(1), "Z": jnp.zeros(1)}


def init_independent_states(key, vars, potential_fn):
    children , tree = jax.tree_util.tree_flatten(vars)
    num_keys = len(children)
    keys = jrandom.split(key, num_keys)

    state = jax.tree_map(lambda x, k: (x,MCMCState(k)), vars, tree.unflatten(keys))
    return state 

def init_joint_state(key, vars):
    children, tree = jax.tree_util.tree_flatten(vars)
    concat = jnp.concatenate(children)
    return concat, MCMCState(key)


state = init_state(key, vars)

def kernelize(f):

    def kernelized_f(x, state, **kwargs):
        (key, num_steps, params), tree = jax.tree_util.tree_flatten(state, is_leaf=lambda x: not isinstance(x, MCMCState))
        key1, key2 = jrandom.split(key)
        x_new = f(x, key1, **params)
        new_state = jax.tree_util.tree_unflatten(tree, (key2, num_steps + 1, params))
        return x_new, new_state

    return kernelized_f

@kernelize
def gaussian_kernel(x, key, step_size = 0.1):
    return x + jrandom.normal(key, shape=x.shape) * step_size


def metropolis_hasting(log_prob, x, x_new, **kwargs):
    log_accept_ratio = log_prob(x_new, **kwargs) - log_prob(x, **kwargs)
    accept = jrandom.uniform(key) < jnp.exp(log_accept_ratio)
    return jnp.where(accept, x_new, x), accept

@partial(jax.jit, static_argnums=(0,))
def update_state(kernel, state):
    out = jax.tree_map(lambda x: kernel(*x), state, is_leaf=lambda x: isinstance(x, tuple))
    # Metripolis Hastings
    log_accept_ratio = 0

    # Parameter updater if adaptive
    return out


@partial(jax.jit, static_argnums=(2,))
def run_mcmc(key, vars, kernel, num_steps):
    state = init_independent_states(key, vars)
    out = jax.lax.fori_loop(0, num_steps, lambda i, x: update_state(kernel, x), state)
    return out



In [409]:
class MCMCKernel:
    _symmetric: bool = True
    _requires_metropolis_hasting: bool = True

    def __call__(self, x, state):
        (key, num_steps, params), tree = jax.tree_util.tree_flatten(state, is_leaf=lambda x: not isinstance(x, MCMCState))
        key1, key2 = jrandom.split(key)
        x_new = self._sample(key1,x, **params)
        new_state = jax.tree_util.tree_unflatten(tree, (key2, num_steps + 1, params))
        return x_new, new_state
    
    def _sample(self, key, x, **params):
        pass

    def log_potential(self, x, **params):
        pass

    def init_state(self, key, **params):
        return MCMCState(key, **params)

class GaussianKernel(MCMCKernel):
    def _sample(self, key, x, step_size = 0.1):
        return x + jrandom.normal(key, shape=x.shape) * step_size
    

kernel = GaussianKernel()
state = init_independent_states(key, vars)

In [412]:
out = run_mcmc(key, vars, kernel, 100)

In [391]:
joint_var, state = init_joint_state(key, vars)

log_prob_fn(joint_var)

TypeError: log_prob_fn() missing 1 required positional argument: 'Z'

In [387]:
flatten_vars, tree = jax.tree_util.tree_flatten(vars)
concat = jnp.concatenate(flatten_vars)
unflatt = jax.tree_util.tree_unflatten(tree, concat)
unflatt

{'X': Array(1., dtype=float32), 'Z': Array(0., dtype=float32)}

In [438]:
class MCMC():

    def __init__(self, kernel, potential_fn, init_vals) -> None:
        
        self.kernel = kernel
        self.potential_fn = potential_fn
        self.init_vals = init_vals

    @partial(jax.jit, static_argnums=(0,))
    def run(self, key, num_steps):
        state = init_independent_states(key, self.init_vals)
        out = jax.lax.fori_loop(0, num_steps, lambda i, x: update_state(self.kernel, x), state)
        return out
    
mcmc = MCMC(kernel, None, vars)

In [446]:
mcmc.run(jrandom.PRNGKey(0), 1000)

{'X': (Array([2.4750533], dtype=float32),
  MCMCState(key=[3835344797  112727439], num_steps=1000)),
 'Z': (Array([-1.1238413], dtype=float32),
  MCMCState(key=[1696239274 2525956687], num_steps=1000))}

In [325]:
print(jax.tree_util.tree_structure(gaussian_kernel))

PyTreeDef(*)


In [327]:
kernels = {"X": gaussian_kernel, "Z": gaussian_kernel}


update_state(kernels, state)

ValueError: Non-hashable static arguments are not supported. An error occurred during a call to 'update_state' while trying to hash an object of type <class 'dict'>, {'X': <function kernelize.<locals>.kernelized_f at 0x7fb25a989900>, 'Z': <function kernelize.<locals>.kernelized_f at 0x7fb25a989900>}. The error was:
TypeError: unhashable type: 'dict'


In [314]:
def update_state2(kernels, state):
    out = jax.tree_map(lambda k,x: k(*x), kernels, state)
    # Metripolis Hastings


    # Parameter updater if adaptive
    return out

In [315]:
kernels = [gaussian_kernel, gaussian_kernel]

In [318]:
run_mcmc(jrandom.PRNGKey(0), [jnp.ones(1), jnp.zeros(1)], [gaussian_kernel, gaussian_kernel], 1000)

ValueError: Non-hashable static arguments are not supported. An error occurred during a call to 'run_mcmc' while trying to hash an object of type <class 'list'>, [<function kernelize.<locals>.kernelized_f at 0x7fb25a98bb50>, <function kernelize.<locals>.kernelized_f at 0x7fb25a98bb50>]. The error was:
TypeError: unhashable type: 'list'


In [271]:
b_vars = {"X": jnp.ones((100,1)), "Z": jnp.zeros((100,1))}
keys = jrandom.split(key, 100)
states = jax.vmap(lambda ks,vs: run_mcmc(ks, vs, gaussian_kernel, 1000))(keys, b_vars)

In [233]:
update_state(gaussian_kernel, state)

{'X': (Array([1.0807748], dtype=float32),
  MCMCState(key=[1278412471 2182328957], num_steps=1)),
 'Z': (Array([0.15642284], dtype=float32),
  MCMCState(key=[1190051861 3378399878], num_steps=1))}

In [211]:
jax.tree_map(gaussian_kernel, vars, state)

ValueError: not enough values to unpack (expected 3, got 1)

In [128]:
state = MCMCState(key, 0, step_size=0.1)
flatten, tree = jax.tree_util.tree_flatten(state, is_leaf=lambda x: not isinstance(x, MCMCState))

In [124]:
tree.children

<bound method PyCapsule.children of PyTreeDef(CustomNode(MCMCState[None], [*, *, {'step_size': *}]))>

In [90]:
jax.tree_map(lambda x,s: x + jrandom.normal(s.key), vars, state)

{'X': Array([1.1438905], dtype=float32),
 'Z': Array([-1.2515389], dtype=float32)}

In [131]:
@kernelize
def gaussian_kernel(x, key, step_size = 0.1):
    return x + jrandom.normal(key) * step_size

In [134]:
gaussian_kernel(jnp.ones(1), state["X"])

TypeError: 'MCMCState' object is not subscriptable

In [367]:
def log_prob_fn(X, Z):
    return jnp.sum(X * Z)

In [ ]:
def flatten_fn(f, )

In [ ]:
def gaussian_kernel(state, vars):
    keys = jax.tree_map(lambda x: x.key, state)
    keys = jrandom.split(keys, num_keys)

    return state, vars